# Exp16: Per-Student Problem Rating Report

This notebook builds a readable report for each student and problem ID. For every problem, it shows:
- the problem description
- the student code, prettified in a formatted code block
- Human A rating
- Human B rating
- LLM_V1 rating
- LLM_V2 rating
- LLM_V3 rating

In [15]:
import json
import os
from html import escape
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)

from lib.experiment_utils import load_best_attempts_df
from utils.constants import PROBLEM_PROMPT_PATH

REPORT_DIR = Path('results/human_validation')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_IDS = ['10155', '14475', '14476']

RATER_FILES = {
    'Human A': {
        '10155': 'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json',
        '14475': 'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json',
        '14476': 'dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json',
    },
    'Human B': {
        '10155': 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json',
        '14475': 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json',
        '14476': 'dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json',
    },
    'LLM_V1': {
        '10155': 'results/human_validation/llm_baseline_annotations_10155.json',
        '14475': 'results/human_validation/llm_baseline_annotations_14475.json',
        '14476': 'results/human_validation/llm_baseline_annotations_14476.json',
    },
    'LLM_V2': {
        '10155': 'results/human_validation/llm_enriched_annotations_v2_10155.json',
        '14475': 'results/human_validation/llm_enriched_annotations_v2_14475.json',
        '14476': 'results/human_validation/llm_enriched_annotations_v2_14476.json',
    },
    'LLM_V3': {
        '10155': 'results/human_validation/llm_v3_annotations_10155.json',
        '14475': 'results/human_validation/llm_v3_annotations_14475.json',
        '14476': 'results/human_validation/llm_v3_annotations_14476.json',
    },
}

def load_annotation_map(filepath: str) -> dict[str, list[str]]:
    path = Path(filepath)
    if not path.exists():
        return {}

    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    annotations: dict[str, list[str]] = {}
    for pid, val in data.get('annotations', {}).items():
        if isinstance(val, dict):
            gaps = val.get('gaps', [])
        elif isinstance(val, list):
            gaps = val
        else:
            gaps = []

        cleaned = []
        for gap in gaps if isinstance(gaps, list) else []:
            gap_text = str(gap).strip()
            if gap_text and gap_text not in cleaned:
                cleaned.append(gap_text)
        annotations[str(pid)] = cleaned

    return annotations

def load_v3_details(filepath: str) -> tuple[dict[str, list[str]], dict[str, str]]:
    path = Path(filepath)
    if not path.exists():
        return {}, {}

    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    annotations: dict[str, list[str]] = {}
    reasoning_map: dict[str, str] = {}
    for pid, val in data.get('annotations', {}).items():
        if isinstance(val, dict):
            gaps = val.get('gaps', [])
        elif isinstance(val, list):
            gaps = val
        else:
            gaps = []

        cleaned = []
        for gap in gaps if isinstance(gaps, list) else []:
            gap_text = str(gap).strip()
            if gap_text and gap_text not in cleaned:
                cleaned.append(gap_text)
        annotations[str(pid)] = cleaned

        raw_response = data.get('raw_responses', {}).get(pid, {})
        parsed_response = raw_response.get('parsed_response', {}) if isinstance(raw_response, dict) else {}
        reasoning_map[str(pid)] = str(parsed_response.get('reasoning', '')).strip()

    return annotations, reasoning_map

def get_problem_meta(problem_id: int, problem_prompts_df: pd.DataFrame) -> dict[str, object]:
    row = problem_prompts_df[problem_prompts_df['ProblemID'] == int(problem_id)]
    if row.empty:
        return {'assignment_id': None, 'requirement': '(problem description unavailable)'}

    item = row.iloc[0]
    assignment_id = item.get('AssignmentID')
    if pd.isna(assignment_id):
        assignment_id = None
    else:
        assignment_id = int(assignment_id)

    return {
        'assignment_id': assignment_id,
        'requirement': str(item.get('Requirement', '(problem description unavailable)')),
    }

def get_best_code(student_id: str, problem_id: int, attempts_df: pd.DataFrame) -> str:
    rows = attempts_df[(attempts_df['SubjectID'] == int(student_id)) & (attempts_df['ProblemID'] == int(problem_id))].copy()
    if rows.empty:
        return ''

    if 'Attempt' in rows.columns:
        rows = rows.sort_values(['Score', 'Attempt'])
    else:
        rows = rows.sort_values(['Score'])

    row = rows.iloc[-1]
    code = row.get('Code', '')
    return '' if pd.isna(code) else str(code)

def get_best_score(student_id: str, problem_id: int, attempts_df: pd.DataFrame) -> float | None:
    rows = attempts_df[(attempts_df['SubjectID'] == int(student_id)) & (attempts_df['ProblemID'] == int(problem_id))].copy()
    if rows.empty:
        return None

    if 'Attempt' in rows.columns:
        rows = rows.sort_values(['Score', 'Attempt'])
    else:
        rows = rows.sort_values(['Score'])

    row = rows.iloc[-1]
    score = row.get('Score', None)
    return None if pd.isna(score) else float(score)

def render_code_block(code: str) -> str:
    text = code if code else '(no code found)'
    return f"<pre class='student-code'><code>{escape(text)}</code></pre>"

def pretty_rating(gaps: list[str] | None) -> str:
    if gaps is None:
        return 'Missing'
    if len(gaps) == 0:
        return 'No gaps'
    return ', '.join(gaps)

def render_gap_badges(gaps: list[str] | None) -> str:
    if gaps is None:
        return "<span class='gap-empty'>Missing</span>"
    if len(gaps) == 0:
        return "<span class='gap-empty'>No gaps</span>"
    badges = ''.join(f"<span class='gap-badge'>{escape(gap)}</span>" for gap in gaps)
    return f"<div class='gap-badge-wrap'>{badges}</div>"

problem_prompts_df = pd.read_csv(PROBLEM_PROMPT_PATH)
best_attempts_df = load_best_attempts_df()

ratings_by_student = {}
v3_reasoning_by_student = {}
for student_id in STUDENT_IDS:
    ratings_by_student[student_id] = {}
    for rater_name, file_map in RATER_FILES.items():
        if rater_name == 'LLM_V3':
            annotations, reasoning_map = load_v3_details(file_map[student_id])
            ratings_by_student[student_id][rater_name] = annotations
            v3_reasoning_by_student[student_id] = reasoning_map
        else:
            ratings_by_student[student_id][rater_name] = load_annotation_map(file_map[student_id])

report_rows = []
skipped_perfect = 0
for student_id in STUDENT_IDS:
    student_problem_ids = set(best_attempts_df[best_attempts_df['SubjectID'] == int(student_id)]['ProblemID'].dropna().astype(int).tolist())
    for source_map in ratings_by_student[student_id].values():
        student_problem_ids.update(int(pid) for pid in source_map.keys())

    for problem_id in sorted(student_problem_ids):
        meta = get_problem_meta(problem_id, problem_prompts_df)
        row = {
            'StudentID': student_id,
            'ProblemID': problem_id,
            'AssignmentID': meta['assignment_id'],
            'ProblemDescription': meta['requirement'],
            'StudentCode': get_best_code(student_id, problem_id, best_attempts_df),
            'Score': get_best_score(student_id, problem_id, best_attempts_df),
        }
        if row['Score'] is not None and row['Score'] >= 1.0:
            skipped_perfect += 1
            continue
        for rater_name, source_map in ratings_by_student[student_id].items():
            row[rater_name] = pretty_rating(source_map.get(str(problem_id))) if str(problem_id) in source_map else 'Missing'
        row['LLM_V3_Reasoning'] = v3_reasoning_by_student.get(student_id, {}).get(str(problem_id), '')
        report_rows.append(row)

report_df = pd.DataFrame(report_rows).sort_values(['StudentID', 'AssignmentID', 'ProblemID'], na_position='last').reset_index(drop=True)
csv_path = REPORT_DIR / 'exp16_student_problem_rating_report.csv'
html_path = REPORT_DIR / 'exp16_student_problem_rating_report.html'
report_df.to_csv(csv_path, index=False)

print(f'Report rows: {len(report_df)}')
print(f'Skipped perfect-score rows: {skipped_perfect}')
print(f'Saved CSV: {csv_path}')
print(f'Saved HTML: {html_path}')

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Report rows: 73
Skipped perfect-score rows: 77
Saved CSV: results/human_validation/exp16_student_problem_rating_report.csv
Saved HTML: results/human_validation/exp16_student_problem_rating_report.html


In [16]:
html_parts = [
    "<style>",
    ".report-wrap { font-family: Georgia, 'Times New Roman', serif; }",
    ".student-card { border: 1px solid #d7dde5; border-radius: 14px; padding: 0.9rem 1rem; margin: 1rem 0; background: linear-gradient(180deg, #ffffff 0%, #fbfcfe 100%); }",
    ".student-card > summary { cursor: pointer; font-size: 1.08rem; font-weight: 700; }",
    ".problem-card { margin-top: 0.75rem; border-top: 1px solid #edf0f5; padding-top: 0.8rem; }",
    ".problem-card > summary { cursor: pointer; font-size: 1rem; font-weight: 700; color: #1f2937; }",
    ".problem-desc { white-space: pre-wrap; line-height: 1.5; color: #1f2937; margin-top: 0.45rem; }",
    ".problem-meta { color: #5b6575; font-size: 0.92rem; margin-top: 0.2rem; }",
    ".problem-score { margin-top: 0.25rem; color: #374151; font-size: 0.92rem; font-weight: 700; }",
    ".reasoning-box { margin-top: 0.35rem; padding: 0.55rem 0.7rem; border-radius: 10px; background: #f8fafc; border: 1px solid #e5e7eb; color: #334155; font-size: 0.9rem; line-height: 1.45; }",
    ".student-code { margin: 0.6rem 0 0.8rem; padding: 0.95rem 1rem; border-radius: 12px; background: #0b1020; color: #e5edf7; overflow-x: auto; white-space: pre; font-size: 0.87rem; line-height: 1.45; }",
    ".gap-badge-wrap { display: flex; flex-wrap: wrap; gap: 0.4rem; margin-top: 0.25rem; }",
    ".gap-badge { display: inline-block; padding: 0.18rem 0.55rem; border-radius: 999px; background: #dbeafe; color: #1d4ed8; font-size: 0.82rem; font-weight: 700; }",
    ".gap-empty { display: inline-block; padding: 0.18rem 0.55rem; border-radius: 999px; background: #f3f4f6; color: #6b7280; font-size: 0.82rem; font-weight: 700; }",
    ".rating-table { width: 100%; border-collapse: collapse; margin-top: 0.5rem; }",
    ".rating-table th, .rating-table td { border: 1px solid #d7dde5; padding: 0.55rem 0.65rem; vertical-align: top; text-align: left; }",
    ".rating-table th { background: #f7f9fc; width: 18%; }",
    ".rating-divider { margin: 0.85rem 0; border: 0; border-top: 1px solid #cfd6e0; }",
    "</style>",
    "<div class='report-wrap'>",
]

for student_id, student_df in report_df.groupby('StudentID', sort=True):
    html_parts.append(f"<details class='student-card' open><summary>Student {escape(str(student_id))} — {len(student_df)} problems</summary>")
    for _, row in student_df.sort_values(['AssignmentID', 'ProblemID'], na_position='last').iterrows():
        assignment_text = f"Assignment {int(row['AssignmentID'])}" if pd.notna(row['AssignmentID']) else 'Assignment unknown'
        problem_key = str(int(row['ProblemID']))
        v3_reasoning = v3_reasoning_by_student.get(student_id, {}).get(problem_key, '')
        rating_rows = ''.join([
            f"<tr><th>Human A</th><td>{render_gap_badges(ratings_by_student[student_id]['Human A'].get(problem_key))}</td></tr>"
            ,f"<tr><th>Human B</th><td>{render_gap_badges(ratings_by_student[student_id]['Human B'].get(problem_key))}</td></tr>"
            ,f"<tr><th>LLM_V1</th><td>{render_gap_badges(ratings_by_student[student_id]['LLM_V1'].get(problem_key))}</td></tr>"
            ,f"<tr><th>LLM_V2</th><td>{render_gap_badges(ratings_by_student[student_id]['LLM_V2'].get(problem_key))}</td></tr>"
            ,f"<tr><th>LLM_V3</th><td>{render_gap_badges(ratings_by_student[student_id]['LLM_V3'].get(problem_key))}<div class='reasoning-box'><strong>Reasoning:</strong> {escape(v3_reasoning) if v3_reasoning else 'No reasoning available'}</div></td></tr>"
        ])
        html_parts.append(
            f"<details class='problem-card'><summary>Problem {int(row['ProblemID'])} — {escape(assignment_text)}</summary><div class='problem-meta'><strong>Problem ID:</strong> {int(row['ProblemID'])} | <strong>{escape(assignment_text)}</strong></div><div class='problem-score'>Student Score: {'' if pd.isna(row['Score']) else f'{float(row['Score']):.3f}'}</div><div class='problem-desc'><strong>Problem Description</strong><br>{escape(str(row['ProblemDescription']))}</div><div style='margin-top: 0.65rem;'><strong>Student Code</strong>{render_code_block(str(row['StudentCode']))}</div><hr class='rating-divider'><table class='rating-table'><tbody>{rating_rows}</tbody></table></details>"
        )
    html_parts.append('</details>')

html_parts.append('</div>')
report_html = ''.join(html_parts)
html_path.write_text(report_html, encoding='utf-8')
display(HTML(report_html))
display(report_df[['StudentID', 'ProblemID', 'AssignmentID', 'Score', 'Human A', 'Human B', 'LLM_V1', 'LLM_V2', 'LLM_V3', 'LLM_V3_Reasoning']].head(12))

Human A,LogicAndNotOrIf/ElseLogicCompareNumLogicBoolean
Human B,If/ElseLogicAndNotOrLogicBoolean
LLM_V1,If/ElseLogicAndNotOrNestedIf
LLM_V2,If/ElseLogicAndNotOr
LLM_V3,"LogicAndNotOrReasoning: The student attempts to handle the two main cases (outsideMode true vs. false) using multiple `else if` branches. However, the conditions within these branches are logically flawed due to incorrect grouping of sub-expressions and a misunderstanding of operator precedence for `&&` and `||`. Specifically, the conditions `n<= 1 || n>= 10 && !outsideMode` and `n<= 1 || n>= 10 && outsideMode` are evaluated as `(n <= 1) || (n >= 10 && ...)` because `&&` has higher precedence than `||`. This leads to incorrect results when `n <= 1` is true, as it makes the entire `||` expression true regardless of the `outsideMode` part. For example, if `n=0` and `outsideMode=false`, the second `else if` condition `(0 <= 1 || 0 >= 10 && !false)` evaluates to `(true || false && true)` which simplifies to `true`, causing the function to incorrectly return `true` when it should be `false`."
Human A,If/ElseLogicAndNotOrLogicBooleanLogicCompareNum
Human B,LogicAndNotOrLogicBooleanIf/Else
LLM_V1,If/ElseLogicAndNotOrLogicBooleanNestedIf
LLM_V2,If/ElseLogicAndNotOrLogicBooleanNestedIf
LLM_V3,"LogicAndNotOrReasoning: The student's code fails to incorporate the `isWeekend` parameter into the conditional logic. The `if` and `else if` statements check cigar counts but do not differentiate between weekend and non-weekend scenarios, leading to incorrect behavior when it's not the weekend and cigars are above 60. Additionally, the student uses redundant `&& true` in conditions and creates dead code with `&& false`, indicating a misunderstanding of how boolean operators combine values. The core issue is the incorrect combination of boolean conditions required by the problem statement."
Human A,If/ElseLogicAndNotOrLogicCompareNum


,StudentID,ProblemID,AssignmentID,Score,Human A,Human B,LLM_V1,LLM_V2,LLM_V3,LLM_V3_Reasoning
0,10155,3,439,0.812500,"LogicAndNotOr, If/Else, LogicCompareNum, Logic...","If/Else, LogicAndNotOr, LogicBoolean","If/Else, LogicAndNotOr, NestedIf","If/Else, LogicAndNotOr",LogicAndNotOr,The student attempts to handle the two main ca...
1,10155,234,439,0.923077,"If/Else, LogicAndNotOr, LogicBoolean, LogicCom...","LogicAndNotOr, LogicBoolean, If/Else","If/Else, LogicAndNotOr, LogicBoolean, NestedIf","If/Else, LogicAndNotOr, LogicBoolean, NestedIf",LogicAndNotOr,The student's code fails to incorporate the `i...
2,10155,235,439,0.857143,"If/Else, LogicAndNotOr, LogicCompareNum","LogicAndNotOr, If/Else, LogicCompareNum","If/Else, LogicAndNotOr, NestedIf","If/Else, LogicAndNotOr, NestedIf","LogicAndNotOr, LogicCompareNum, If/Else","The student's code has several issues. First, ..."
3,10155,236,439,0.750000,"LogicAndNotOr, If/Else, LogicCompareNum",LogicCompareNum,"If/Else, LogicAndNotOr, NestedIf","If/Else, LogicAndNotOr, NestedIf","LogicAndNotOr, If/Else",The student's code attempts to handle three mu...
4,10155,22,487,0.360000,"LogicCompareNum, NestedIf, LogicAndNotOr, If/Else","DefFunction, NestedIf, LogicCompareNum","DefFunction, For, If/Else, LogicAndNotOr, Logi...","DefFunction, If/Else, LogicAndNotOr, LogicComp...","DefFunction, Math+-*/, LogicCompareNum, LogicA...",The student's `noTeenSum` method fails to call...
5,10155,24,487,0.590909,"If/Else, LogicCompareNum, Math+-*/, LogicAndNotOr","LogicCompareNum, Math+-*/, If/Else","If/Else, LogicAndNotOr, LogicBoolean, LogicCom...","DefFunction, If/Else, LogicAndNotOr, LogicComp...","LogicCompareNum, If/Else",The student's code attempts to use an if-else ...
6,10155,28,487,0.200000,"LogicCompareNum, StringIndex, If/Else, LogicAn...","LogicAndNotOr, LogicCompareNum, StringFormat, ...","If/Else, LogicBoolean, StringConcat, StringIndex","If/Else, StringConcat, StringIndex, StringLen","LogicCompareNum, StringIndex, StringConcat, St...","The student's code has several issues. First, ..."
7,10155,100,487,0.071429,"If/Else, DefFunction, Math%, Math+-*/, LogicCo...","DefFunction, If/Else, LogicAndNotOr, LogicComp...","DefFunction, If/Else, Math%","DefFunction, If/Else, Math%, Math+-*/","DefFunction, If/Else, Math+-*/, Math%, LogicCo...",The student's submission demonstrates a fundam...
8,10155,101,487,0.000000,No gaps,No gaps,No gaps,No gaps,No gaps,The student's submission is a trivial placehol...
9,10155,102,487,0.538462,No gaps,No gaps,No gaps,No gaps,No gaps,"The student's code is a trivial placeholder, s..."
